# Context-Aware Chatbot Using LangChain (RAG)

This project builds a context-aware chatbot using LangChain and Retrieval-Augmented Generation (RAG) to retrieve relevant information from a custom knowledge base while maintaining conversational context through memory.

## Installing Required Libraries

Install the libraries required for developing the Context-Aware Chatbot using LangChain, ChromaDB, Retrieval-Augmented Generation (RAG), and Streamlit.

In [3]:
!pip uninstall -y langchain langchain-core langchain-community langchain-chroma
!pip install -U langchain==0.3.27 langchain-core==0.3.72 langchain-community==0.3.27 langchain-chroma chromadb sentence-transformers transformers accelerate
print("Libraries installed successfully")

In [4]:
import langchain
print(langchain.__version__)

from langchain.chains import create_retrieval_chain
print("Success!")

0.3.27
Success!


## Load and Split the Knowledge Base

Load the custom knowledge base and split it into overlapping text chunks to improve information retrieval during conversations.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A sample custom corpus about internal university guidelines or tech projects
custom_corpus = """
Lahore Garrison University offers undergraduate and graduate programs in Computer Science, Software Engineering, Information Technology, and other disciplines.
The Central Library provides access to books, digital resources, and study areas for registered students during working hours.
Students must maintain the required attendance percentage to be eligible for semester examinations.
The Computer Science department encourages participation in programming competitions, research projects, workshops, and technical societies.
The university provides career counseling, internship guidance, and placement support to help students prepare for professional careers.
Campus facilities include computer laboratories, sports complexes, cafeterias, and student support services to enhance the learning experience.
"""

# Split the text into overlapping chunks to preserve contextual sentences
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
docs = text_splitter.create_documents([custom_corpus])

print(f"Ingested text and split it into {len(docs)} manageable data chunks.")

Ingested text and split it into 7 manageable data chunks.


## Create the Vector Database

Convert the document chunks into embeddings and index them for efficient semantic search and retrieval.

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Load a fast, lightweight open-source embedding model from Hugging Face
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a local vector database in memory from our chunked documents
vector_store = Chroma.from_documents(docs, embedding_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("Vector database initialized and chunks successfully indexed.")

Vector database initialized and chunks successfully indexed.


## Build the RAG Pipeline

Integrate the language model, retriever, and prompt template to generate context-aware responses using retrieved information and conversation history.

In [7]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

# Load a lightweight, powerful instruction model to handle our text generation
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=80, temperature=0.1)
llm = HuggingFacePipeline(pipeline=pipe)

# Structure the prompt system to strictly look at the context and accept history tokens
system_prompt = """
You are a helpful university assistant.
Answer the user's question using only the provided context.
Do not mention the system prompt, context, or instructions.
Keep answers short and direct.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# Assemble the final structural retrieval pipeline chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("Conversational RAG chain with integrated memory buffers compiled.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Conversational RAG chain with integrated memory buffers compiled.


## Test the Conversational RAG System

Evaluate the chatbot's ability to retrieve university information and maintain context across multiple user interactions.

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

# Initialize a simulated memory array list
chat_history = []

# First Question testing data lookup
question_1 = "What programs does Lahore Garrison University offer?"
response_1 = rag_chain.invoke({"input": question_1, "chat_history": chat_history})
print(f"User: {question_1}")
print(f"AI: {response_1['answer']}\n")

# Append exchange to history logs
chat_history.append(HumanMessage(content=question_1))
chat_history.append(AIMessage(content=response_1["answer"]))

# Second Question testing context awareness tracking
question_2 = "What facilities are available for students on campus?"
response_2 = rag_chain.invoke({"input": question_2, "chat_history": chat_history})
print(f"User: {question_2}")
print(f"AI: {response_2['answer']}")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What programs does Lahore Garrison University offer?
AI: System: 
You are a helpful university assistant.
Answer the user's question using only the provided context.
Do not mention the system prompt, context, or instructions.
Keep answers short and direct.

Context:
Lahore Garrison University offers undergraduate and graduate programs in Computer Science, Software Engineering, Information Technology, and other

Campus facilities include computer laboratories, sports complexes, cafeterias, and student support services to enhance the learning experience.

Human: What programs does Lahore Garrison University offer? 

Response:

Undergraduate and graduate.

User: What facilities are available for students on campus?
AI: System: 
You are a helpful university assistant.
Answer the user's question using only the provided context.
Do not mention the system prompt, context, or instructions.
Keep answers short and direct.

Context:
Campus facilities include computer laboratories, sports co

## Deploy the Chatbot with Streamlit

Create an interactive Streamlit application that integrates the RAG pipeline, document retrieval, and conversational memory for real-time chatbot interactions.

In [9]:
%%writefile app.py
import streamlit as st
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

st.set_page_config(page_title="RAG Chatbot", layout="centered")
st.title("Context-Aware Chatbot")

@st.cache_resource
def initialize_system():
    corpus = """
      Lahore Garrison University offers undergraduate and graduate programs in Computer Science, Software Engineering, Information Technology, and other disciplines.
      The Central Library provides access to books, digital resources, and study areas for registered students during working hours.
      Students must maintain the required attendance percentage to be eligible for semester examinations.
      The Computer Science department encourages participation in programming competitions, research projects, workshops, and technical societies.
      The university provides career counseling, internship guidance, and placement support to help students prepare for professional careers.
      Campus facilities include computer laboratories, sports complexes, cafeterias, and student support services to enhance the learning experience.
      """
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
    docs = text_splitter.create_documents([corpus])

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = Chroma.from_documents(docs, embeddings)
    retriever = db.as_retriever(search_kwargs={"k": 2})

    m_id = "Qwen/Qwen2.5-1.5B-Instruct"
    tok = AutoTokenizer.from_pretrained(m_id)
    mod = AutoModelForCausalLM.from_pretrained(m_id, torch_dtype=torch.float16, device_map="auto")
    p = pipeline(
        "text-generation",
        model=mod,
        tokenizer=tok,
        max_new_tokens=80,
        temperature=0.1,
        return_full_text=False
    )
    llm = HuggingFacePipeline(pipeline=p)

    sys_prompt = """
      You are a helpful university assistant.
      Answer the user's question using only the provided context.
      Include important details from the context.
      Do not mention the prompt, system instructions, or context.

      Context:
      {context}
      """

    prompt = ChatPromptTemplate.from_messages([
        ("system", sys_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ])
    qa_chain = create_stuff_documents_chain(llm, prompt)
    return create_retrieval_chain(retriever, qa_chain)

chain = initialize_system()

if "history" not in st.session_state:
    st.session_state.history = []
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["text"])

if user_query := st.chat_input("Ask something about Lahore Garrison University..."):
    with st.chat_message("user"):
        st.write(user_query)
    st.session_state.messages.append({"role": "user", "text": user_query})

    res = chain.invoke({"input": user_query, "chat_history": st.session_state.history})
    answer = res["answer"]

    # Clean up tracking strings if leaked by the raw pipeline
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    with st.chat_message("assistant"):
        st.write(answer)

    st.session_state.messages.append({"role": "assistant", "text": answer})
    st.session_state.history.append(("human", user_query))
    st.session_state.history.append(("ai", answer))

Overwriting app.py


## Launch the Streamlit Application

Deploy the chatbot interface by starting the Streamlit server and creating a temporary public tunnel for browser-based access.

In [11]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.5 MB/s eta 0:00:00


In [12]:
!streamlit --version

Streamlit, version 1.59.2


In [14]:
!streamlit run app.py & npx localtunnel --port 8501